### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](../resources/monkey.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

%matplotlib inline

In [2]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
DATA_DIR = "/data/bojkoar/nlp/week02_lm"
os.makedirs(DATA_DIR, exist_ok=True)
ARCHIVE_PATH = os.path.join(DATA_DIR, "arxivData.json.tar.gz")
!wget -x "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O {ARCHIVE_PATH}
!tar -xvzf {ARCHIVE_PATH}
data = pd.read_json(ARCHIVE_PATH)
data.sample(n=5)

--2026-10-11 16:28:09--  https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.70.18, 2620:100:6026:18::a27d:4612
Connecting to www.dropbox.com (www.dropbox.com)|162.125.70.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1 [following]
--2026-10-11 16:28:09--  https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc097d2516dfc866512b7b24393e.dl.dropboxusercontent.com/cd/0/inline/DJyXqkNBmQqTtXeW2dQZfhrc4aSbiLQPKsRxiwG3-QcgKcynJ7-XVyAiftWrGqLQYp7qWTHo-giyCxJjn0aVccBbuBSd46YdCUE2fjNENgMl79h6y_tkxXl6RwUkBL6gP3s/file?dl=1# [following]
--2026-10-11 16:28:10--  https://uc097d2516dfc866512b7b24393e.dl.dropbox

,author,day,id,link,month,summary,tag,title,year
26275,"[{'name': 'Xiang Yang'}, {'name': 'Peter Meer'}]",20,1609.06371v3,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,The robust estimator presented in this paper p...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Robust Estimation of Multiple Inlier Structures,2016
12758,"[{'name': 'Bilal Abbasi'}, {'name': 'Jeff Cald...",15,1608.04348v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",8,"Nondominated sorting, also called Pareto Depth...","[{'term': 'cs.LG', 'scheme': 'http://arxiv.org...",Anomaly detection and classification for strea...,2016
27557,"[{'name': 'Lachlan Tychsen-Smith'}, {'name': '...",30,1703.10295v3,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,We define the object detection from imagery pr...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",DeNet: Scalable Real-time Object Detection wit...,2017
13844,"[{'name': 'H. Sharifi Noghabi'}, {'name': 'H. ...",8,1510.02513v2,"[{'rel': 'related', 'href': 'http://dx.doi.org...",10,Differential Evolution (DE) is one of the most...,"[{'term': 'cs.NE', 'scheme': 'http://arxiv.org...",A novel mutation operator based on the union o...,2015
5972,"[{'name': 'David Lopez-Paz'}, {'name': 'Suvrit...",1,1402.0119v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,Classical methods such as Principal Component ...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Randomized Nonlinear Component Analysis,2014


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer

from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()
lines = [" ".join(tokenizer.tokenize(text.lower())) for text in tqdm(lines)]

sorted(lines, key=len)[:3]

100%|██████████| 41000/41000 [00:28<00:00, 1445.40it/s]


['differential contrastive divergence ; this paper has been retracted .',
 'what does artificial life tell us about death ? ; short philosophical essay',
 'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .']

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:

$$
    P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1})
$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$
    P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})
$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [6]:
from tqdm import tqdm
from collections import defaultdict, Counter
from typing import Iterable

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines: Iterable[str], n: int):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - every prefix must contain exactly (n - 1) tokens. If a prefix is shorter than that,
      pad it with UNK on the left. The number of padding tokens therefore depends on n:
      derive it from n, do not hard-code it. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
      For n=2 the very same prefixes give (UNK,), (the,) and (approach,), and for n=1 all
      of them give the empty tuple (). Equivalently: prepend (n - 1) UNK tokens to a line,
      then slide a window of size n over it.
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.

    Useful invariant: a line with L tokens always contributes exactly L + 1 counted events
    (its L tokens plus EOS), for every n. Note also that NGramLanguageModel below already
    implements this padding rule in get_possible_next_tokens - keep the two consistent, or
    the model will look up prefixes that you never counted.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for text in tqdm(lines, "Calculating n-grams", unit="tok"):
        tokens = [UNK] * (n-1) + text.split() + [EOS]
        for *prefix, next_token in zip(*(tokens[i:] for i in range(n))):
            counts[tuple(prefix)][next_token] += 1

    return counts


In [7]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

# the padding rule must hold for every n, not just for n=3
for n in (1, 2, 3, 5, 10):
    counts_n = count_ngrams(dummy_lines, n=n)
    assert set(map(len, counts_n.keys())) == {n - 1}, \
        "every prefix must have exactly (n - 1) tokens, got lengths %s for n=%i" % (
            set(map(len, counts_n.keys())), n)
    assert sum(sum(c.values()) for c in counts_n.values()) == 2877, \
        "each line must contribute exactly len(tokens) + 1 events (its tokens plus EOS) " \
        "for every n. If this passes for n=3 but fails otherwise, you have most likely " \
        "hard-coded the number of UNK padding tokens instead of deriving it from n."

Calculating n-grams: 100%|██████████| 100/100 [00:00<00:00, 18117.94tok/s]


Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [8]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)
        for prefix in counts:
            for next_token in counts[prefix]:
                self.probs[prefix][next_token] = counts[prefix][next_token] / counts[prefix].total()

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [9]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Calculating n-grams: 100%|██████████| 100/100 [00:00<00:00, 21734.40tok/s]


Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [10]:
lm = NGramLanguageModel(lines, n=3)

Calculating n-grams:   2%|▏         | 1019/41000 [00:00<00:16, 2413.64tok/s]

Calculating n-grams: 100%|██████████| 41000/41000 [00:18<00:00, 2195.54tok/s]


The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [11]:
import numpy as np
from numpy.random import default_rng, Generator

def sample_with_temperature(dist: dict[str, float], temperature = 1.0, rng: Generator = None):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    if rng is None:
        rng = default_rng()
    possible_tokens = list(dist.keys())
    power = 1 if temperature == 0.0 else (1 / temperature)
    probs = np.array([dist[next_token] ** power for next_token in possible_tokens])
    probs /= np.sum(probs)
    if temperature == 0.0:
        max_prob = np.max(probs)
        count = np.sum(probs == max_prob)
        probs[probs != max_prob] = 0.0
        probs[probs != 0.0] = 1.0 / count
    return rng.choice(possible_tokens, p=probs)

In [12]:
def nucleus_filter(dist: dict[str, float], nucleus=0.9):
    """Keep the smallest high-probability token set with cumulative mass >= nucleus."""
    dist_list = sorted(dist.items(), key=lambda i: (-i[1], i[0]))
    result: dict[str, float] = {}
    total_p = 0.0
    for token, p in dist_list:
        result[token] = p
        total_p += p
        if total_p >= nucleus: break
    return result


def sample_nucleus(dist: dict[str, float], nucleus=0.9, rng=None):
    """Sample one token after applying nucleus filtering."""
    if rng is None:
        rng = default_rng()
    dist = nucleus_filter(dist, nucleus=nucleus)
    possible_tokens = list(dist.keys())
    probs = np.array(list(dist.values()))
    probs /= np.sum(probs)
    return rng.choice(possible_tokens, p=probs)

In [13]:
from numpy.random import choice

def get_next_token(lm: NGramLanguageModel, prefix: tuple, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    # return sample_with_temperature(
    #     {next_token: lm.get_next_token_prob(prefix, next_token) for next_token in lm.get_possible_next_tokens(prefix)},
    #     temperature=temperature
    # )
    return sample_nucleus(
        {next_token: lm.get_next_token_prob(prefix, next_token) for next_token in lm.get_possible_next_tokens(prefix)},
        nucleus=0.99
    )

In [14]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
print(test_freqs)
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

# test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
# assert 1500 < test_freqs['learning'] < 3000
# test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
# assert 8000 < test_freqs['learning'] < 9000
# test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
# assert test_freqs['learning'] == 10000

print("Looks nice!")

Counter({np.str_('been'): 9216, np.str_('not'): 349, np.str_('only'): 171, np.str_('also'): 118, np.str_('occurred'): 81, np.str_('lately'): 65})
Looks nice!


Let's have fun with this model

In [15]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial intelligence community , various methods use a quick way to shorten proofs as programs . we find that the proposed and implemented with $ w $ so that each model is then used for collocation extraction approach based on genetic linkage analysis tasks , present a learnability result applies directly to identify the new modality ( e . g . smart augmentation and marginalization techniques . we obtain the optimal solution allows easily deploying this system , which has the same fidelity ) then predicts the class of predictors and deep learning ; sequence - to - sequence learning ;


In [16]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the gap between the nn by formulating the problem of denoising and is able to serve as a consequence , the roi size , in the amount of redundancy , we first train their filters . however , the density estimate . our proposed darkrank method on two real applications . we demonstrate the value of 26 submissions across 3 evaluation scenarios . in particular , on the acl2 system . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $-\frac1N$, where $N$ is the __total number of scored tokens__ in the corpora.

A sentence $w_1 \dots w_L$ contributes $L + 1$ factors to that product, because EOS is predicted and scored exactly like a regular token:

$$P(w_1 \mid \varnothing) \cdot P(w_2 \mid w_1) \cdots P(w_L \mid w_1 \dots w_{L-1}) \cdot P(EOS \mid w_1 \dots w_L)$$

Here $\varnothing$ is the empty prefix. EOS is counted in $N$ as well, so $N = \sum_{\text{sentences}} (L + 1)$ -- the same number of events you counted in `count_ngrams`.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [30]:
np.exp(-0.2)

np.float64(0.8187307530779818)

In [43]:
def perplexity(lm: NGramLanguageModel, lines: list[str], min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: scoring a line "w_1 w_2 ... w_L" means scoring L + 1 events:
        P(w_1 | ""), P(w_2 | "w_1"), ..., P(w_L | "w_1 ... w_{L-1}"), P(EOS | "w_1 ... w_L")
    The prefix of the very first token is the empty string "", i.e. an empty prefix and not
    a string of UNKs: the model pads it to (n - 1) UNK tokens on its own inside
    get_possible_next_tokens, so do not add any padding here. Likewise, w_1 is simply the
    first real token of the line. The trailing EOS is scored like any other token and is
    counted in N too, so a line with L tokens adds L + 1 to N.

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    log_sum = 0.0
    N = 0
    for text in lines:
        tokens = text.split() + [EOS]
        N += len(tokens)
        for i in range(len(tokens)):
            prob = lm.get_next_token_prob(" ".join(tokens[:i]), tokens[i])
            log_sum += max(np.log(prob), min_logprob)
    log_sum *= (-1 / N)

    return np.exp(log_sum)

In [44]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

Calculating n-grams: 100%|██████████| 100/100 [00:00<00:00, 11412.76tok/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184



/tmp/ipykernel_33421/682777773.py:24: RuntimeWarning: divide by zero encountered in log
  log_sum += max(np.log(prob), min_logprob)


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [ ]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


In [ ]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [ ]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [ ]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

In [ ]:
# optional: try to sample tokens from such a model

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $\lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1. Discounting takes $\delta$ away from each of the $N_{1+}(prefix_{n-1}, \bullet)$ __distinct__ tokens ever seen after that prefix, and that is exactly the mass to redistribute:
$$\lambda_{prefix_{n-1}} = {\delta \cdot N_{1+}(prefix_{n-1}, \bullet) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)}$$
- $P_{kn}(w_t | prefix_{n-2})$ is the lower-order model: Kneser-Ney smoothing for the {n-1}-gram language model, i.e. the same formula with the oldest token dropped from the prefix.
- Unigram $P_{kn}(w_t)$ is a special case that terminates the recursion: how likely it is to see $w_t$ in an unfamiliar context

__Prefixes that were never seen in training.__ If $Count(prefix_{n-1}, \hat w) = 0$ for every $\hat w$, then the denominator is zero and both the first term and $\lambda_{prefix_{n-1}}$ come out as $\tfrac{0}{0}$. There is simply no higher-order evidence to interpolate with, so back off completely: take $\lambda_{prefix_{n-1}} = 1$ and the first term equal to $0$, which means returning $P_{kn}(w_t | prefix_{n-2})$ unchanged. The result is still a valid probability distribution, and this is the case you will hit constantly at test time.

__Choosing $\delta$.__ Keep $\delta \in (0, 1]$. Counts are integers $\ge 1$, so within that range $\max(0, Count - \delta) = Count - \delta$ and the probabilities sum to exactly 1. With $\delta > 1$ the $\max$ clips away less mass than $\lambda$ hands back to the lower order, and the distribution stops normalizing -- the validity test below will fail.

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

_Note: the test below iterates over `dummy_lm.vocab`, so your class has to expose a `self.vocab` set holding every known token, just like `LaplaceLanguageModel` does._

In [ ]:
class KneserNeyLanguageModel(NGramLanguageModel):
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        # hint: you will need n-gram counts for every order from 1 to n, plus a self.vocab set
        <YOUR CODE>

    def get_possible_next_tokens(self, prefix):
        < YOUR CODE >

    def get_next_token_prob(self, prefix, next_token):
        # remember to handle a prefix that never occurred in training, see above
        <YOUR CODE>

In [ ]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=<...>)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))